# klee100 Qwen3.8-Flash-Next AutoRound 3bpw + MTP (without the BF16 PLE shard) -> private Kaggle Model (run this ON Kaggle)

**Notebook settings: Accelerator = None (CPU), Internet = ON.** No GPU quota is used. Utility notebook: no
competition data, never submitted.

### Before running: attach two Kaggle Secrets, start the run from the editor
Add-ons -> Secrets -> attach `KAGGLE_USERNAME` and `KAGGLE_KEY` (account **foysalemonshanto**), then
**Save Version -> Save & Run All** from the notebook editor (a run started through the Kaggle API could not reach
the secrets service on 2026-09-26). The account check refuses any other user before any transfer.

**Private storage:** the Model is private and needs about 50.7 GB of free private Kaggle storage on
foysalemonshanto. The instance is created only after the transfer, so a full quota fails at that step (E11-01 on
2026-10-15 streamed 19.5 GB and then failed on the quota).

### How it works (no disk): the same tested code as the ARC-AGI-3 NVIDIA upload (132.7 GB, 2026-09-26) and the FP8 PLE overlay (51.2 GB, 2026-10-05)
Every file streams straight from Hugging Face into Kaggle's resumable upload URL in 256 MiB chunks (RAM only,
3 workers). Each file is hashed while it streams and compared with the pinned Hugging Face tree. The model
instance is created **only if all 32 files match**. A dropped connection resumes from the byte the server
confirmed. The last cell is fail-closed: it raises unless Kaggle lists every file, including the mirror manifest,
with the expected size.

### What gets published (50.68 GB, 32 files + `KAGGLE_MIRROR_MANIFEST.json`)
`klee100/Qwen3.8-Flash-Next-AutoRound-3bpw-MTP` at revision `600715b31b`, byte-identical. **No fixkeys:** config.json, quantization_config.json and the
index are the original files.
- 10 main shards `model-00002..00011-of-00011.safetensors`: AutoRound SignRoundV2, auto_round:auto_gptq mixed bits.
  Routed experts are W3A16G128 on layers 0-11 and 36-47 and W2A16G64 on layers 12-35, the main projections are
  W8A16G128, and the sensitive paths are BF16.
- 2 MTP shards `mtp-model-0000{1,2}-of-00002.safetensors` (W4A16G128 experts).
- `model.safetensors.index.json`, unchanged. Its 128 PLE `ngram_embedding` entries still point to the shard that is
  not uploaded.
- `config.json`, `quantization_config.json`, `generation_config.json`.
- Tokenizer, processor and template files: `tokenizer.json`, `tokenizer_config.json`, `vocab.json`, `merges.txt`,
  `chat_template.jinja`, and the processor / preprocessor / video preprocessor configs.
- `build_info.json`, `mtp_pack_plan.json`, the quality / serving / source-validation / validation reports, LICENSE
  and README.md.

**Not uploaded:**
- `model-00001-of-00011.safetensors` (102.4 GB): the BF16 PLE table. It holds exactly 128 `ngram_embedding` tensors and nothing
  else, by both the index and the shard header.
- `.gitattributes`.

At run time the GPU notebook merges these files with the 10 FP8 PLE shards of `foysalemonshanto/qwen38-flash-next-fp8ple-overlay` and the fixkeys
config.

License: Qwen Community License 1.0 (LICENSE file included). Kaggle label "Other (specified in description)", the
same label the NVIDIA mirror and the FP8 PLE overlay were accepted with.

Kaggle path afterwards: `/kaggle/input/models/foysalemonshanto/qwen38-flash-next-klee100-3bpw-mtp/pytorch/autoround-3bpw-mtp-nople/1`

The run is idempotent: if the model instance already exists it only lists and verifies its files.


In [ ]:
import hashlib
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

HF_REPO = "klee100/Qwen3.8-Flash-Next-AutoRound-3bpw-MTP"
HF_REVISION = "600715b31bed3833fb4a51d9ea5d684ea2c7f32c"   # pinned 2026-10-31 (HF main, last modified 2026-09-24)
HF_URL = "https://huggingface.co/{repo}/resolve/{rev}/{name}"
KAGGLE_MODEL_SLUG = "qwen38-flash-next-klee100-3bpw-mtp"
KAGGLE_MODEL_TITLE = "Qwen3.8 Flash-Next klee100 AutoRound 3bpw MTP"
KAGGLE_FRAMEWORK = "pyTorch"          # appears as "pytorch" in the /kaggle/input path
KAGGLE_INSTANCE_SLUG = "autoround-3bpw-mtp-nople"
LICENSE_NAME = "Other (specified in description)"   # Qwen Community License 1.0; label accepted for the NVIDIA mirror
PLACEHOLDER_DIR = Path("/tmp/klee100-instance-placeholders")   # empty files only: names for model_instance_create
CHUNK = 256 << 20          # bytes per upload request (multiple of 256 KiB, as the resumable protocol requires)
QUANTUM = 256 << 10
READ = 8 << 20             # Hugging Face read size
WORKERS = 3

# name -> (bytes, hash kind, hash). "sha256" = LFS oid (x-linked-etag) for LFS files, sha256 of the content for
# the small files (the streamer always computes sha256). Taken from the Hugging Face tree + HEAD at HF_REVISION.
# Not uploaded: model-00001-of-00011.safetensors (BF16 PLE table only) and .gitattributes.
EXPECTED_FILES = {
    'LICENSE': (3235, 'sha256', 'a0dc422560841fd68e06d974907f8b4c709bca44a67daad2b528437bdf676c08'),
    'README.md': (2591, 'sha256', '0c523e37ea7c94d37a70de5e3b60e038af26d72d12bb892240e64e2be8e28674'),
    'build_info.json': (1303, 'sha256', 'e568181a667b620012aea5e948490b467cd62e39eedfe1ab3c1cc8d99b216e8d'),
    'chat_template.jinja': (8952, 'sha256', 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041'),
    'config.json': (4863351, 'sha256', 'fb17316df83cc1cf6cb4f5336fefb33566102cfeda16d1e9fc89ae1a4c61b9ce'),
    'generation_config.json': (219, 'sha256', '1a450b75a54bf9e6ffdc681d8b764d453061dd2d271df33fcfed46949b67c3de'),
    'merges.txt': (3353259, 'sha256', 'a9d356d7bdf1ef4949e3e748e95b8e10ad9d4e2e838eddc38a0a7b6b94d1db8d'),
    'model-00002-of-00011.safetensors': (5001197632, 'sha256', 'a4b0226cbe33061ac782da081e8c18f7bba713c4e502abc514948650e7438c3c'),
    'model-00003-of-00011.safetensors': (5002786464, 'sha256', 'ca6a60732d654fe9478fab222a9c3278db6cf4e351449813a4c9ffe8d9dde615'),
    'model-00004-of-00011.safetensors': (5002313320, 'sha256', '14e89121d36cd139392df6c6918e4ec211ff41421224d9a417cff5da3d89d4f9'),
    'model-00005-of-00011.safetensors': (5003380328, 'sha256', 'e0cbe52f256c6e4c69e82c1d23f87eeff769da3d65aad02a9d1d56a895619c24'),
    'model-00006-of-00011.safetensors': (5003576312, 'sha256', 'e6fa4479f32d6219d8c50f45054bde16d1be493bd5b043c270eb0ab2fac0c9ef'),
    'model-00007-of-00011.safetensors': (5003604352, 'sha256', '75d1b488b74154ae1d7d26b4a7681eec805ce523223629f3dd06f8476937fbae'),
    'model-00008-of-00011.safetensors': (5003576312, 'sha256', '8008bff489bea0444e9d86e86a085e1b6218fd2a98ab0cfa905cfec9d363e109'),
    'model-00009-of-00011.safetensors': (5002896008, 'sha256', '0634fc00117e9c13e38db58d1a59a6371b64227e87eb3f3dcfdca1aa7b05cd14'),
    'model-00010-of-00011.safetensors': (4996759496, 'sha256', '2cec1f796afc231b00fa8a61d41de96ea3de508e71688a3618309af77b765617'),
    'model-00011-of-00011.safetensors': (4166074128, 'sha256', '823f6e7632d9c82c138ed57a0a2eeed44560d0dfacdf53d95b9abcbf5f772ab6'),
    'model.safetensors.index.json': (24122451, 'sha256', '873191b509904c7007ee9a791ecc18a939afbf11797e1549c0e3279bf31cefd4'),
    'mtp-model-00001-of-00002.safetensors': (527759104, 'sha256', 'cc34aee13fcc259728a4019b1e3c609823eb25786adb5ac73a258b57245f2780'),
    'mtp-model-00002-of-00002.safetensors': (898326280, 'sha256', '5d2e1e267b7d61967b3324399fd4962cc3734a3c83f930649a15f2c05f3be76c'),
    'mtp_pack_plan.json': (70870, 'sha256', '9c3f4b4d9b2065cd6a35f38ab71aa8005210c127c16bbfa60307cb9830eabc98'),
    'preprocessor_config.json': (443, 'sha256', '3a159dfec9978a186a72ba085e0ad6a050f3968d8b364218d7bd13f5c89381f2'),
    'processor_config.json': (1220, 'sha256', 'bfbc24af59a3e73a9cd0653b8d4ae758dfaec4e3e6c15dfb1c6c8ee8d5683c85'),
    'quality_report.json': (1303, 'sha256', 'bf63b011eaeed54818bd5ef1a9850ff7815b4af6b641a382408c45d125c1465f'),
    'quantization_config.json': (4533673, 'sha256', '876b577c3f790db22df92f9dbcad0f02a7b341fa6b1e4686591d001a5acaa3a9'),
    'serving_report.json': (2120, 'sha256', 'dd43660247f0664630ac3030ed85b4d1aa53d90623730b52761f6b28560ad0b0'),
    'source_validation.json': (6243, 'sha256', 'e37552c6a930e733e8c222006f366e0fe8ab8c21648fea3c99937606776b82d0'),
    'tokenizer.json': (19989325, 'sha256', '06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523'),
    'tokenizer_config.json': (1165, 'sha256', '792fa3f0cb88b111e54ef3134c873531008c4df471d108da17903426e308aa7b'),
    'validation_report.json': (1471, 'sha256', '725e8e81b25911497471deb8cb26f4f46b0cfca7343eef5ced94047386b8a337'),
    'video_preprocessor_config.json': (385, 'sha256', '7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13'),
    'vocab.json': (6722759, 'sha256', 'ce99b4cb2983d118806ce0a8b777a35b093e2000a503ebde25853284c9dfa003'),
}
EXPECTED_TOTAL_BYTES = 50_675_936_074
NOT_UPLOADED = set()
assert sum(v[0] for v in EXPECTED_FILES.values()) == EXPECTED_TOTAL_BYTES
assert len(EXPECTED_FILES) == 32
print("python", sys.version.split()[0], "| cpus", os.cpu_count(),
      "| /tmp free (df, not the session quota):", round(shutil.disk_usage("/tmp").free / 1e9, 1), "GB")


In [ ]:
# Kaggle Secrets -> Kaggle API credentials. Stops here (before any transfer) if they are missing or wrong.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME").strip()
    _key = _secrets.get_secret("KAGGLE_KEY").strip()
except Exception as exc:
    raise RuntimeError(
        f"Could not read Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY ({exc!r}). Attach both (Add-ons -> Secrets) "
        "and start the run from the notebook editor (Save Version -> Save & Run All). A run started through the "
        "Kaggle API got ConnectionError from the secrets service on 2026-09-26.") from None

cred_dir = Path.home() / ".kaggle"
cred_dir.mkdir(parents=True, exist_ok=True)
(cred_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _key}))
(cred_dir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = _key
del _key

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.4"])

from kaggle.api.kaggle_api_extended import KaggleApi
from kaggle.models.upload_file import UploadFile
from kagglesdk.blobs.types.blob_api_service import ApiBlobType, ApiStartBlobUploadRequest

api = KaggleApi()
api.authenticate()
if KAGGLE_USERNAME != "foysalemonshanto":
    raise RuntimeError(f"secrets belong to {KAGGLE_USERNAME!r}, expected foysalemonshanto - nothing transferred")
MODEL_REF = f"{KAGGLE_USERNAME}/{KAGGLE_MODEL_SLUG}"
INSTANCE_REF = f"{MODEL_REF}/pytorch/{KAGGLE_INSTANCE_SLUG}"

# Credential check first: kernels_list(mine=True) answers 401 for a wrong key. (Get-calls cannot be used for
# this: Kaggle answers 403 PERMISSION_DENIED both for a missing model and for someone else's private one.)
try:
    api.kernels_list(mine=True, page_size=1)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    raise RuntimeError(f"Kaggle rejected the KAGGLE_USERNAME / KAGGLE_KEY secrets (HTTP {status}). "
                       "Check the two secret values, then run again.") from None

model_exists = MODEL_REF in {getattr(m, "ref", None) for m in (api.model_list(owner=KAGGLE_USERNAME, page_size=100) or [])}
instance_exists = False
if model_exists:
    try:
        api.model_instance_get(INSTANCE_REF)
        instance_exists = True
    except Exception as exc:
        if getattr(getattr(exc, "response", None), "status_code", None) not in (403, 404):
            raise
print(f"Kaggle API OK as {KAGGLE_USERNAME}; model exists={model_exists}; instance exists={instance_exists}")
if instance_exists:
    print("The instance already exists: the transfer is skipped; the last cell lists its files.")


In [ ]:
# Stream every file Hugging Face -> Kaggle resumable upload, hashing on the fly. Nothing is written to disk.
class _HFReader:
    """Reads one file from a byte offset in a background thread into a bounded queue (<= 48 x 8 MiB)."""

    def __init__(self, url, start, size):
        self.q = queue.Queue(maxsize=48)
        self.stop = threading.Event()
        self.error = None
        self.thread = threading.Thread(target=self._run, args=(url, start, size), daemon=True)
        self.thread.start()

    def _run(self, url, start, size):
        try:
            headers = {"Accept-Encoding": "identity"}
            if start:
                headers["Range"] = f"bytes={start}-"
            with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if start and r.status_code != 206:
                    raise RuntimeError(f"Hugging Face ignored the Range header (HTTP {r.status_code})")
                got = start
                for piece in r.iter_content(READ):
                    if self.stop.is_set():
                        return
                    got += len(piece)
                    self.q.put(piece)
                if got != size:
                    raise RuntimeError(f"Hugging Face stream ended at {got} of {size} bytes")
        except BaseException as exc:
            self.error = exc
        finally:
            self.q.put(None)

    def get(self):
        piece = self.q.get()
        if piece is None:
            raise self.error or RuntimeError("Hugging Face stream ended early")
        return piece

    def close(self):
        self.stop.set()
        try:
            while True:
                self.q.get_nowait()
        except queue.Empty:
            pass


def _start_blob(name, size):
    req = ApiStartBlobUploadRequest()
    req.type = ApiBlobType.MODEL
    req.name = name
    req.content_length = size
    req.last_modified_epoch_seconds = int(time.time())
    with api.build_kaggle_client() as kaggle:
        resp = api.with_retry(kaggle.blobs.blob_api_client.start_blob_upload)(req)
    return resp.token, resp.create_url


def _persisted(response, size):
    """Bytes the upload server has stored, from a chunk or status response."""
    if response.status_code in (200, 201):
        return size
    if response.status_code == 308:
        rng = response.headers.get("Range")
        return int(rng.rsplit("-", 1)[1]) + 1 if rng else 0
    raise RuntimeError(f"upload server answered HTTP {response.status_code}: {response.text[:300]}")


def _query(session, put_url, size):
    r = session.put(put_url, headers={"Content-Range": f"bytes */{size}", "Content-Length": "0"}, timeout=(30, 120))
    return _persisted(r, size)


def stream_file(name, size, kind, max_stalls=12):
    """Returns (token, sha256 hex, git-sha1 hex or None, seconds, failures). Gives up only after max_stalls
    consecutive failures without progress (a 53.7 GB file is ~200 requests; isolated resets are expected)."""
    token, put_url = _start_blob(name, size)
    session = requests.Session()
    h256 = hashlib.sha256()
    gsha = hashlib.sha1(b"blob %d\0" % size) if kind == "gitsha1" else None
    hashed = persisted = 0          # hashed == bytes confirmed stored and folded into the digests
    failures = stalls = 0
    stall_mark = -1
    next_report = 5e9
    started = time.time()
    while hashed < size or persisted < size:
        reader = _HFReader(HF_URL.format(repo=HF_REPO, rev=HF_REVISION, name=name), hashed, size)
        buf = bytearray()
        buf_start = hashed
        try:
            while buf_start < size:
                while len(buf) < CHUNK and buf_start + len(buf) < size:
                    buf += reader.get()
                if persisted > buf_start:          # stored by an earlier attempt: hash only, do not resend
                    n = min(persisted - buf_start, len(buf))
                else:
                    at_end = buf_start + len(buf) == size
                    send = len(buf) if at_end else (len(buf) // QUANTUM) * QUANTUM
                    r = session.put(put_url, data=bytes(buf[:send]), timeout=(30, 900),
                                    headers={"Content-Range": f"bytes {buf_start}-{buf_start + send - 1}/{size}"})
                    persisted = _persisted(r, size)
                    n = persisted - buf_start
                    if n < 0 or n > len(buf):
                        raise RuntimeError(f"server reports {persisted} stored; chunk started at {buf_start}")
                with memoryview(buf) as mv:
                    h256.update(mv[:n])
                    if gsha is not None:
                        gsha.update(mv[:n])
                del buf[:n]
                buf_start += n
                hashed = buf_start
                if hashed >= next_report:
                    dt = time.time() - started
                    print(f"    {name}: {hashed / 1e9:.1f}/{size / 1e9:.1f} GB ({hashed / 1e6 / dt:.0f} MB/s)", flush=True)
                    next_report += 5e9
        except Exception as exc:
            failures += 1
            stalls = stalls + 1 if hashed == stall_mark else 1
            stall_mark = hashed
            print(f"    {name}: attempt failed at {hashed} bytes (stall {stalls}/{max_stalls}): {exc!r}"[:400], flush=True)
            if stalls >= max_stalls:
                raise
            session.close()
            session = requests.Session()          # drop pooled connections after a failure
            time.sleep(min(120, 10 * stalls))
            for q in range(5):
                try:
                    persisted = _query(session, put_url, size)
                    break
                except Exception as qexc:
                    if q == 4:
                        raise
                    print(f"    {name}: status query failed ({qexc!r}); retrying"[:300], flush=True)
                    time.sleep(15 * (q + 1))
            if persisted < hashed:
                raise RuntimeError(f"{name}: server now reports {persisted} bytes < {hashed} already confirmed")
        finally:
            reader.close()
    return token, h256.hexdigest(), gsha.hexdigest() if gsha is not None else None, time.time() - started, failures


def upload_bytes(name, data):
    token, put_url = _start_blob(name, len(data))
    r = requests.put(put_url, data=data, headers={"Content-Range": f"bytes 0-{len(data) - 1}/{len(data)}"},
                     timeout=(30, 300))
    if _persisted(r, len(data)) != len(data):
        raise RuntimeError(f"{name}: small upload incomplete")
    return token


TOKENS, HASHES = {}, {}
if not instance_exists:
    todo = sorted((n for n in EXPECTED_FILES if n not in NOT_UPLOADED), key=lambda n: -EXPECTED_FILES[n][0])
    t0 = time.time()

    def _one(name):
        size, kind, want = EXPECTED_FILES[name]
        print(f"[{time.strftime('%H:%M:%S')}] start {name} ({size / 1e9:.2f} GB)", flush=True)
        for attempt in range(1, 4):      # a fresh upload session per attempt (e.g. if the old one expired)
            try:
                token, s256, g1, dt, fails = stream_file(name, size, kind)
                break
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] {name}: file attempt {attempt}/3 failed: {exc!r}"[:400], flush=True)
                if attempt == 3:
                    raise
        got = s256 if kind == "sha256" else g1
        print(f"[{time.strftime('%H:%M:%S')}] done  {name} in {dt:.0f} s ({size / 1e6 / max(dt, 1e-6):.0f} MB/s, "
              f"{fails} retries) {kind} {'OK' if got == want else 'MISMATCH'}", flush=True)
        return name, token, s256, got == want

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for name, token, s256, ok in pool.map(_one, todo):
            TOKENS[name], HASHES[name] = token, (s256, ok)
    bad = sorted(n for n, (_, ok) in HASHES.items() if not ok)
    total = sum(EXPECTED_FILES[n][0] for n in todo)
    dt = time.time() - t0
    print(f"\nstreamed {len(todo)} files, {total / 1e9:.2f} GB in {dt / 60:.1f} min ({total / 1e6 / dt:.0f} MB/s)")
    if bad:
        raise RuntimeError(f"hash mismatch, instance NOT created: {bad}")
    print(f"all {len(todo)} files match the pinned Hugging Face revision byte-for-byte")

    mirror_manifest = {
        "source": {"hf_repo": HF_REPO, "hf_revision": HF_REVISION},
        "verified_against": "Hugging Face tree API at the pinned revision (LFS sha256 / git blob sha1), "
                            "hashed while streaming",
        "files": {n: {"bytes": EXPECTED_FILES[n][0], "sha256": HASHES[n][0]} for n in sorted(todo)},
        "total_bytes": total,
        "not_uploaded": sorted(NOT_UPLOADED),
        "serving_notes": {
            "what": "klee100 3bpw-MTP checkpoint without its BF16 PLE shard; merge at run time with the 10 "
                    "model-plefp8 shards of foysalemonshanto/qwen38-flash-next-fp8ple-overlay and the fixkeys config",
            "not_uploaded": {
                "model-00001-of-00011.safetensors": {"bytes": 102400512288, "sha256": "64836044b0d06b54a3a1f3fd3cf4ce06ba005fb6319abb100479b7a151f7f44c",
                    "why": "BF16 PLE table only: 128 ple.ple_embedding.ngram_embedding tensors, nothing else"},
                ".gitattributes": "git metadata",
            },
            "index": "original model.safetensors.index.json; its 128 ngram_embedding entries point to "
                     "model-00001-of-00011.safetensors, which the runtime graft replaces with the FP8 PLE overlay",
            "configs": "original config.json and quantization_config.json (no fixkeys applied)",
            "quantization": "AutoRound SignRoundV2, auto_round:auto_gptq mixed: routed experts W3A16G128 (layers "
                            "0-11, 36-47) and W2A16G64 (layers 12-35), main projections W8A16G128, BF16 sensitive "
                            "paths, MTP experts W4A16G128",
            "runtime": "Merlin v3.0.0 (humming kernels for the W2/W3 experts, Marlin for W4/W8), fixkeys overlay",
            "license": "Qwen Community License 1.0 (see LICENSE)",
        },
    }
    MANIFEST_BYTES = (json.dumps(mirror_manifest, indent=1) + "\n").encode("utf-8")
    TOKENS["KAGGLE_MIRROR_MANIFEST.json"] = upload_bytes("KAGGLE_MIRROR_MANIFEST.json", MANIFEST_BYTES)


In [ ]:
# Publish: parent model (if new), then the instance, handing the already-uploaded blob tokens to the library.
if not instance_exists:
    if not model_exists:
        parent_dir = Path("/tmp/kaggle-model-parent")
        shutil.rmtree(parent_dir, ignore_errors=True)
        parent_dir.mkdir(parents=True)
        (parent_dir / "model-metadata.json").write_text(json.dumps({
            "ownerSlug": KAGGLE_USERNAME,
            "title": KAGGLE_MODEL_TITLE,
            "slug": KAGGLE_MODEL_SLUG,
            "subtitle": "klee100 AutoRound 3bpw MTP of Qwen3.8-Flash-Next, without the BF16 PLE shard",
            "isPrivate": True,
            "description": (
                f"Private mirror of {HF_REPO} (revision {HF_REVISION}) for offline Kaggle inference, without "
                "model-00001-of-00011.safetensors (the BF16 PLE table): 10 main shards + 2 MTP shards + index + configs + "
                "tokenizer files + reports, byte-identical. Merge with the FP8 PLE shards of "
                "foysalemonshanto/qwen38-flash-next-fp8ple-overlay. License: Qwen Community License 1.0 (LICENSE in the files)."
            ),
            "licenseName": LICENSE_NAME,
        }, indent=2))
        created = api.model_create_new(str(parent_dir))
        if getattr(created, "error", None):
            raise RuntimeError(f"model create failed: {created.error}")
        print("model created:", getattr(created, "ref", None), getattr(created, "url", None))

    # model_instance_create walks a folder and asks _upload_file for a token per file. The folder holds empty
    # placeholders with the real names; the patched _upload_file returns the token of the streamed blob.
    shutil.rmtree(PLACEHOLDER_DIR, ignore_errors=True)
    PLACEHOLDER_DIR.mkdir(parents=True)
    for name in TOKENS:
        (PLACEHOLDER_DIR / name).touch()
    (PLACEHOLDER_DIR / "model-instance-metadata.json").write_text(json.dumps({
        "ownerSlug": KAGGLE_USERNAME,
        "modelSlug": KAGGLE_MODEL_SLUG,
        "instanceSlug": KAGGLE_INSTANCE_SLUG,
        "framework": KAGGLE_FRAMEWORK,
        "overview": (f"{HF_REPO} at {HF_REVISION} without model-00001-of-00011.safetensors (BF16 PLE), uploaded "
                     "byte-identical (hashes in KAGGLE_MIRROR_MANIFEST.json). 10 main + 2 MTP shards, original "
                     "index and configs (no fixkeys)."),
        "usage": ("Symlink these files together with the 10 model-plefp8 shards of foysalemonshanto/qwen38-flash-next-fp8ple-overlay into one "
                  "folder, graft the PLE entries of the index and apply the fixkeys config, then serve with Merlin "
                  "v3.0.0. License: Qwen Community License 1.0 (LICENSE)."),
        "licenseName": LICENSE_NAME,
        "fineTunable": False,
        "trainingData": [],
        "modelInstanceType": "Unspecified",
        "baseModelInstance": "",
        # "" exactly as in the proven Qwen/Occamy notebooks: the URL field belongs to the EXTERNAL_VARIANT type.
        "externalBaseModelUrl": "",
    }, indent=2))
    used = []
    _original_upload_file = KaggleApi._upload_file

    def _token_for(self, file_name, full_path, blob_type, upload_context, quiet, resources, content_type=None):
        if file_name not in TOKENS:
            raise RuntimeError(f"no streamed blob for {file_name}")
        used.append(file_name)
        return UploadFile(token=TOKENS[file_name])

    KaggleApi._upload_file = _token_for
    try:
        response = api.model_instance_create(str(PLACEHOLDER_DIR), quiet=True, dir_mode="skip")
    finally:
        KaggleApi._upload_file = _original_upload_file
    if sorted(used) != sorted(TOKENS):
        raise RuntimeError(f"token hand-off mismatch: used={sorted(used)}")
    if getattr(response, "error", None):
        raise RuntimeError(f"instance create failed: {response.error}")
    print(f"instance created with {len(used)} files:", getattr(response, "ref", None), getattr(response, "url", None))


In [ ]:
# List what Kaggle now holds (processing can lag a few minutes behind the create call).
expected_upload = {n: EXPECTED_FILES[n][0] for n in EXPECTED_FILES if n not in NOT_UPLOADED}
# the manifest is checked too: exact size when this run uploaded it, presence when the instance already existed
expected_upload["KAGGLE_MIRROR_MANIFEST.json"] = len(MANIFEST_BYTES) if "MANIFEST_BYTES" in globals() else None
listed = {}
for attempt in range(20):
    try:
        resp = api.model_instance_version_files(f"{INSTANCE_REF}/1", page_size=100)
        listed = {f.name: f.size for f in (getattr(resp, "files", None) or [])}
    except Exception as exc:
        print("listing not ready:", repr(exc)[:200])
    if set(expected_upload) <= set(listed):
        break
    time.sleep(60)
print(f"{len(listed)} files listed for {INSTANCE_REF}/1")
for name in sorted(listed):
    print(f"  {listed[name]:>14,}  {name}")
missing = sorted(set(expected_upload) - set(listed))
wrong = {n: (listed[n], expected_upload[n]) for n in expected_upload
         if n in listed and expected_upload[n] is not None and listed[n] != expected_upload[n]}
print("missing:", missing or "none", "| size mismatches:", wrong or "none")
if missing or wrong:
    raise RuntimeError(f"Kaggle mirror verification FAILED: missing={missing} wrong_sizes={wrong} - do not attach this model")
print(f"VERIFIED: all {len(expected_upload)} files listed with the expected sizes")
print("\nattach with:", f'"model_sources": ["{INSTANCE_REF}/1"]')
print("mount path :", f"/kaggle/input/models/{INSTANCE_REF}/1")
